# Using AutoModel

In [ ]:
# !pip install transformers

In [ ]:
# Import necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Set the random seed for reproducibility
torch.random.manual_seed(0)

In [ ]:
# Load the pre-trained language model from Hugging Face
# This model is optimized for causal language modeling
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",   # comment to use CPU
    torch_dtype="auto",     # Automatically selects the appropriate data type
    # trust_remote_code=True, # Allows downloading code from the repository
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# Load the tokenizer associated with the model
# Tokenizer converts text into token IDs that the model can understand
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct",device_map="cuda")

In [ ]:
# Define the conversation messages in a list of dictionaries
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "Can you provide ways to eat combinations of bananas and dragonfruits?"},
    {"role": "assistant", "content": "Sure! Here are some ways to eat bananas and dragonfruits together: 1. Banana and dragonfruit smoothie: Blend bananas and dragonfruits together with some milk and honey. 2. Banana and dragonfruit salad: Mix sliced bananas and dragonfruits together with some lemon juice and honey."},
    {"role": "user", "content": "What about solving a 2x + 3 = 7 equation?"},
]

In [ ]:
tokenizer("What about solving a 2x + 3 = 7 equation?")

{'input_ids': [1724, 1048, 17069, 263, 29871, 29906, 29916, 718, 29871, 29941, 353, 29871, 29955, 6306, 29973], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
def predict(input_prompt):
    # Define the conversation context for the model, including system and user roles
    messages = [
        {"role": "system", "content": "You are a helpful assistant that assists users to find the correct methods/approach for security within an organization."},
        {"role": "user", "content": input_prompt}  # The user's input prompt is added here
    ]

    # Format the input message into the appropriate structure for the model using the tokenizer
    text = tokenizer.apply_chat_template(
        messages,  # The structured message for the assistant
        tokenize=False,  # Don't tokenize yet, just prepare the chat template
        add_generation_prompt=True  # Add any additional generation prompt required by the model
    )

    # Tokenize the message text into model input format
    model_inputs = tokenizer([text], return_tensors="pt")
    model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}

    # Generate model output by passing the tokenized input to the model
    # 'max_new_tokens' ensures that the model generates up to a maximum of 4096 new tokens
    generated_ids = model.generate(**model_inputs, max_new_tokens=4096,temperature=0.2)

    # Slice the generated sequence to remove the original input tokens (we only want the output)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs["input_ids"], generated_ids)]

    # Decode the generated token IDs back into a human-readable string, skipping special tokens
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Return the generated response
    return response

In [ ]:
print(predict("What about solving a 2x + 3 = 7 equation?"))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


To solve the linear equation 2x + 3 = 7, follow these steps:

1. Subtract 3 from both sides of the equation to isolate the term with the variable (x):

   2x + 3 - 3 = 7 - 3
   2x = 4

2. Divide both sides of the equation by the coefficient of the variable (2) to solve for x:

   2x / 2 = 4 / 2
   x = 2

So, the solution to the equation 2x + 3 = 7 is x = 2.


# Using pipeline

In [ ]:
# Using the text2text-generation pipeline for translation
# Load a different model for translating Hindi to English
translation_pipe = pipeline("text2text-generation", model="snehalyelmati/mt5-hindi-to-english")

Device set to use cuda:0


In [ ]:
# Translate sentences from Hindi to English
print(translation_pipe("आप कैसे है?")[0]['generated_text'])
print(translation_pipe("मैं कल तुमसे मिलने आऊंगा")[0]['generated_text'])

How are you?
I'll be coming to see you tomorrow.
